In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Chargement de l'artefact N°2
input_path = "artifacts/02_labeled_ml_table.parquet"
df = pd.read_parquet(input_path)
print(f"Dataset étiqueté chargé : {df.shape[0]} lignes.")

# Assurer le format datetime sur la date de commande
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# 2. Plage temporelle
min_date = df['order_purchase_timestamp'].min()
max_date = df['order_purchase_timestamp'].max()
print(f"\n=== PLAGE TEMPORELLE DES COMMANDES ===")
print(f"Date de début : {min_date}")
print(f"Date de fin   : {max_date}")

Dataset étiqueté chargé : 96478 lignes.

=== PLAGE TEMPORELLE DES COMMANDES ===
Date de début : 2016-09-15 12:16:38
Date de fin   : 2018-08-29 15:00:37


In [4]:
# 3. Option A : TIME-BASED SPLIT (70% Train, 15% Val, 15% Test)
# Tri chronologique des commandes
df_sorted = df.sort_values('order_purchase_timestamp').reset_index(drop=True)

n = len(df_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_sorted.iloc[:train_end].copy()
val_df = df_sorted.iloc[train_end:val_end].copy()
test_df = df_sorted.iloc[val_end:].copy()

# 4. Affichage des statistiques par ensemble
print("\n=== STATISTIQUES DU SPLIT TEMPOREL ===")
splits = [('Train', train_df), ('Validation', val_df), ('Test', test_df)]

for label, split_df in splits:
    count = len(split_df)
    pct = (count / n) * 100
    late_ratio = split_df['is_late'].mean() * 100
    d_min = split_df['order_purchase_timestamp'].min().strftime('%Y-%m-%d')
    d_max = split_df['order_purchase_timestamp'].max().strftime('%Y-%m-%d')
    print(f"{label:>10} | Lignes: {count:>6} ({pct:4.1f}%) | Retards: {late_ratio:5.2f}% | Période: {d_min} au {d_max}")


=== STATISTIQUES DU SPLIT TEMPOREL ===
     Train | Lignes:  67534 (70.0%) | Retards:  9.03% | Période: 2016-09-15 au 2018-04-15
Validation | Lignes:  14472 (15.0%) | Retards:  5.34% | Période: 2018-04-15 au 2018-06-21
      Test | Lignes:  14472 (15.0%) | Retards:  6.61% | Période: 2018-06-21 au 2018-08-29


In [5]:
# 5. Sauvegarde des Artefacts
os.makedirs("artifacts", exist_ok=True)
train_df.to_parquet("artifacts/03_train.parquet", index=False)
val_df.to_parquet("artifacts/03_val.parquet", index=False)
test_df.to_parquet("artifacts/03_test.parquet", index=False)

print("\n✅ Artefacts N°3 sauvegardés avec succès dans 'artifacts/' :")
print("  - artifacts/03_train.parquet")
print("  - artifacts/03_val.parquet")
print("  - artifacts/03_test.parquet")


✅ Artefacts N°3 sauvegardés avec succès dans 'artifacts/' :
  - artifacts/03_train.parquet
  - artifacts/03_val.parquet
  - artifacts/03_test.parquet
